In [19]:
#Modules
%config InlineBackend.figure_format='retina'
import numpy as np
np.set_printoptions(threshold=np.inf)
import matplotlib as mpl
import matplotlib.pyplot as plt
from astropy.io import fits
import pandas as pd
from io import StringIO
from matplotlib.path import Path
import os
from astropy.wcs import WCS
from astropy import units as u
from astropy.coordinates import SkyCoord
from shapely.geometry import Polygon
from astropy.utils.data import get_pkg_data_filename
from astropy.wcs import Wcsprm
from astropy.io import fits
from astropy.wcs import utils
from skimage.feature import blob_dog, blob_log, blob_doh
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import colors
from skimage.color import rgb2gray, rgb2hsv, hsv2rgb
from skimage.io import imread, imshow
from sklearn.cluster import KMeans
from astropy.visualization import make_lupton_rgb
from astropy.visualization import SqrtStretch,LogStretch
from astropy.visualization import ZScaleInterval
from reproject import reproject_interp
from astropy.visualization import make_lupton_rgb
#import sep
from matplotlib.patches import Ellipse
from scipy import stats
from scipy.ndimage import gaussian_filter
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from scipy.interpolate import griddata

from scipy.stats import kde
from scipy.ndimage import gaussian_filter
from astropy.io import ascii
from astropy.visualization import simple_norm
from photutils.psf import extract_stars


from astropy.table import Table
from astropy.nddata import NDData

from shapely.geometry import Polygon, Point
from photutils.detection import find_peaks

from skimage.measure import EllipseModel

from astropy.stats import SigmaClip

from photutils.background import Background2D, MedianBackground
from scipy import interpolate
ell = EllipseModel()

image_lims=ZScaleInterval()

plt.rcParams['font.size'] = 15
from scipy.interpolate import LinearNDInterpolator
from scipy.interpolate import CloughTocher2DInterpolator

from astropy.modeling.models import Gaussian2D
from astropy.modeling.models import custom_model
from astropy.modeling import models, fitting
from scipy.optimize import curve_fit

from photutils.segmentation import SegmentationImage
from astropy.stats import sigma_clipped_stats, SigmaClip
from photutils.segmentation import detect_threshold, detect_sources

from astropy.convolution import convolve, Gaussian2DKernel, Gaussian1DKernel
from astropy.convolution import interpolate_replace_nans

from photutils.segmentation import deblend_sources, detect_threshold, detect_sources

from photutils.segmentation import SourceFinder
from photutils.segmentation import SourceCatalog

from astropy.modeling.fitting import LevMarLSQFitter
from astropy.stats import gaussian_sigma_to_fwhm
from photutils.background import MADStdBackgroundRMS, MMMBackground
from photutils.detection import IRAFStarFinder
#from photutils.psf import (DAOGroup, IntegratedGaussianPRF,
#                           IterativelySubtractedPSFPhotometry,BasicPSFPhotometry,prepare_psf_model)

from photutils.datasets import make_noise_image
from photutils.psf import EPSFBuilder
from photutils.utils import circular_footprint
from tempfile import TemporaryFile
from astropy.io import fits as pyfits
import pickle

import math
from statistics import mean
import copy


In [21]:
#Import contour vertices

#Use IRAC coordinates
with open(r"C:\Users\jacob\AstronomyResearchStuff\jw_verts3_conservative.pkl", 'rb') as fp1:
    conservativeVertices = pickle.load(fp1)
    
#Use VMC coordinates
with open(r"C:\Users\jacob\AstronomyResearchStuff\verts_conservative_higher_6MARCH.pkl", 'rb') as fp2:
    conservativeHigherVertices = pickle.load(fp2)

#Use IRAC coordinates
with open(r"C:\Users\jacob\AstronomyResearchStuff\jw_verts3_expansive.pkl", 'rb') as fp3:
    expansiveVertices = pickle.load(fp3)

#Use VMC coordinates
with open(r"C:\Users\jacob\AstronomyResearchStuff\verts_expansive_higher_6MARCH.pkl", 'rb') as fp4:
    expansiveHigherVertices = pickle.load(fp4)

In [23]:
#Load image files and set an extend value
brightPrime = np.load(r"C:\Users\jacob\AstronomyResearchStuff\MW_removed_image_2025_artifact_blocked.npy")

usefulImagePrime = brightPrime

extend = 5

In [25]:
#Open a VMC file for coordinate conversions
hdux = fits.open(r"C:\Users\jacob\AstronomyResearchStuff\e20230725_00130000217_dp_st_tl.fit") 

In [27]:
#Note: the code is currently not configured to create plots associated with the area ratios
#However, cutouts showing pixels considered for each ratio are computed to assist with plotting efforts if so desired

In [29]:
#Define the function used to compute the area ratios using unscaled light-weighted ellipses
def UnscaledRatios(vertices, image, extend):

    #Calculates the isophote's light ellipse-related area ratios using an unscaled light ellipse 

    #Parameters
    #----------

    #vertices: list
        # the vertices of the isophote of interest

    #image: numpy.ndarray
        # the image data that computations are performed on

    #extend: int
        #value used to extend the cutout around an isophote's boundaries
        #5 is a suitable value for these calculations

    #Returns
    #---------

    #ratio1: int
        #the ratio between the ellipse's area outside of the contour to the total area of the ellipse

    #ratio2: int
        #the ratio between the contour's area not within the ellipse to the total area of the contour


    #ratio3: int
        #the ratio between the contour's area included within the ellipse to the total area of the contour

    #Set booleans to assist with debugging
    debug = 0
    debug2 = 0

    #First, make zeros array, where all pixels outside isophote boundaries are set to zero

    #Find the maximum and minimum pixels for the image

    shape = image.shape
    

    xmax,xmin,ymax,ymin = (shape[1]-1),0,(shape[0]-1),0
    
    #Change how the vertices are read in depending on if they are part of an higher set of contours
    if verticesType == 0:

        ra = vertices[0]
        dec = vertices[1]

        xx, yy = WCS(hdux[1].header).all_world2pix(ra,dec, 1)
    
    elif verticesType == 1:
        
        
        pixx1 = vertices[:,0]
        pixy1 = vertices[:,1]

    
        ra, dec = WCS(hdux[1].header).all_pix2world(pixx1,pixy1, 1)
        xx, yy = WCS(hdux[1].header).all_world2pix(ra,dec, 1)

    else:
        print('This is not a valid verticesType value.')
        sys.exit()

    ymax1=max(yy)
    ymin1=min(yy)
    xmax1=max(xx)
    xmin1=min(xx)

    if ymax1 < 0:
        ymax1 = 0
    if ymin1 < 0:
        ymin1 = 0
    if xmax1 < 0:
        xmax1 = 0
    if xmin1 < 0:
        xmin1 =0

    if ymax1 > ymax:
        ymax1 = ymax
    if xmax1 > xmax:
        xmax1 = xmax

    #Set the pixel boundaries around the isophote
    ymax2 = int(np.round(ymax1)) + extend
    ymin2 = int(np.round(ymin1)) - extend
    xmax2 = int(np.round(xmax1)) + extend
    xmin2 = int(np.round(xmin1)) - extend
   
    #Create a cutout around isophote
    cutout = image[ymin2:ymax2,xmin2:xmax2]

    poly_verts = np.vstack((xx-xmin2,yy-ymin2)).T

    shape = cutout.shape
    xnew,ynew = shape[1],shape[0]
    x,y = np.meshgrid(np.arange(xnew),np.arange(ynew))
    x, y = x.flatten(), y.flatten()

    points = np.vstack((x,y)).T
    path = Path(poly_verts)
    grid = path.contains_points(points)

    zeros = np.zeros(cutout.shape)

    grid = grid.reshape((ynew,xnew))

    zeros[grid]=cutout[grid]

    flat_array = cutout[grid]

    array = zeros
    
    #Second, find light-weighted radius

    #note: the array will be handed in as a function of index, with each corresponding to one pixel. The origin of the indexing will always be at a corner. Which corner depends on how you ask the plot to display.
    #Step 1: compute normalization, N.
    N = np.sum(zeros)
    #Step 2: compute light-weighted center.
    ny, nx = zeros.shape
    #Now create an array of x, y that is the pixel centers in the "absolute" coordinates in which it was handed in.
    x = np.arange(0.5, nx, 1)
    y = np.arange(0.5, ny, 1)
    #Now compute light-weighted centers.
    x0, y0 = np.sum(np.tensordot(x, zeros, axes = (0,1)))/N, np.sum(np.tensordot(y, zeros, axes = (0,0)))/N
    

    #Now compute Delta x and Delta y from these centers.
    Delta_x = x - x0
    Delta_y = y - y0
    #Now compute <Delta x>, <Delta y>, other guys from these centers.
    Delta_x_bar = np.sum(np.tensordot(Delta_x, array, axes = (0,1)))/N
    Delta_y_bar = np.sum(np.tensordot(Delta_y, array, axes = (0,0)))/N
    Delta_x_abs_bar = np.sum(np.tensordot(abs(Delta_x), array, axes = (0,1)))/N
    Delta_y_abs_bar = np.sum(np.tensordot(abs(Delta_y), array, axes = (0,0)))/N
    if debug:
        print('Delta_x_abs_bar, Delta_y_abs_bar = ', Delta_x_abs_bar, Delta_y_abs_bar)

    Delta_x_sq_bar = np.sum(np.tensordot(Delta_x*Delta_x, array, axes = (0,1)))/N
    Delta_y_sq_bar = np.sum(np.tensordot(Delta_y*Delta_y, array, axes = (0,0)))/N
    if debug:
        print('Delta_x_sq_bar, Delta_y_sq_bar = ', Delta_x_sq_bar, Delta_y_sq_bar)

    eps = 1e-13

    r_bar = 0.
    r_sq_bar = 0.
    cos_theta_bar = 0.
    sin_theta_bar = 0.
    tan_theta_bar = 0.
    theta_bar = 0.
    dxdy_bar = 0.

    i = 0
    for dyv in Delta_y:
        j = 0
        for dxv in Delta_x:
            if debug2:
                print('i, j =', i, j)
            if debug:
                print('i=, j=, dxv, dyv', i, j, dxv, dyv)
            r_sq = dxv*dxv + dyv*dyv
            r = np.sqrt(r_sq)
            dxdy = dxv*dyv

            #For cosine theta and sine theta.
            if r > eps:
                cos_theta = dxv/r
                #print('i=, j=, cos theta =', i, j, cos_theta)
                sin_theta = dyv/r
            else:
                cos_theta = dxv/eps
                #print('in eps area, i=, j=, cos theta =', i, j, cos_theta)
                sin_theta = dyv/eps
            #For tan theta.
            if dxv > eps:
                tan_theta = dyv/dxv
                theta = math.atan2(dyv, dxv)
                if debug2:
                    print('i=, j=,  theta =', i, j, theta)
                    print('tan theta =', tan_theta)
            else:
                tan_theta = dyv/eps
                theta = math.atan2(dyv, eps)
                if debug2:
                    print('in eps area, i=, j=, theta =', i, j,theta)
                    print('in eps area, tan_theta =', tan_theta)

            #Now apply weighting.
            r_sq *= array[i, j]
            r *= array[i, j]
            cos_theta *= array[i, j]
            #print('weighted cos theta =', cos_theta)
            sin_theta *= array[i, j]
            tan_theta *= array[i, j]
            if debug2:
                print('weighted tan theta =', theta)
            theta *= array[i, j]
            #print('weighted theta =', theta)
            #Now take averages of these obervables.
            r_sq_bar = r_sq_bar + r_sq
            r_bar = r_bar + r
            cos_theta_bar = cos_theta_bar + cos_theta
            sin_theta_bar = sin_theta_bar + sin_theta
            tan_theta_bar = tan_theta_bar + tan_theta
            theta_bar = theta_bar + theta
            dxdy_bar = dxdy_bar + dxdy

            j += 1
        i += 1

    #Now normalize all the above by N.
    r_sq_bar /= N
    r_bar /= N
    cos_theta_bar /= N
    sin_theta_bar /= N
    tan_theta_bar /= N
    theta_bar /= N
    dxdy_bar /= N

    if debug2:
        print('<r^2> =', r_sq_bar, '<r> =', r_bar,'<cos> = ',cos_theta_bar,'<sin> = ',sin_theta_bar, '<tan> = ',tan_theta_bar, '<theta> = ', theta_bar)
    #return [N, x0, y0, Delta_x_bar, Delta_y_bar, Delta_x_abs_bar, Delta_y_abs_bar, Delta_x_sq_bar, Delta_y_sq_bar, r_sq_bar, r_bar, cos_theta_bar, sin_theta_bar, tan_theta_bar, theta_bar ]
    #return r_sq_bar, r_bar #, cos_theta_bar, sin_theta_bar, tan_theta_bar, theta_bar

    xgrid,ygrid = np.meshgrid(x,y)

    #put xgrid and ygrid to have center at x0,y0
    xs = xgrid-x0
    ys = ygrid-y0

    zerosx = np.zeros(cutout.shape)
    zerosx[grid]=xs[grid]

    zerosy = np.zeros(cutout.shape)
    zerosy[grid]=ys[grid]

    # calculate inertia tensor
    T = np.zeros((2,2))
    T[0,0]=np.sum(zerosx**2*zeros)/np.sum(zeros)
    T[0,0]
    T[0,1]=np.sum(zerosx*zerosy*zeros)/np.sum(zeros)
    T[1,0]=T[0,1]
    T[1,1]=np.sum(zerosy*zerosy*zeros)/np.sum(zeros)

    eigenvalues,eigenvectors = np.linalg.eig(T)

    theta = np.linspace(0, 2*np.pi, 1000)
    ellipsis = (np.sqrt(eigenvalues) * eigenvectors) @ [np.sin(theta), np.cos(theta)]

    #print('eigenvectors ',eigenvectors)

    #first, find larger eigenvalue
    loc = np.argsort(eigenvalues)[::-1]

    major,minor = np.sqrt(eigenvalues[loc])

    # u represents axis of major axis
    u = np.array([1,0])
    d = eigenvectors[loc[0]]
    sign = np.sign(eigenvectors[1])

    if sign[0] == -1 and sign[1] == 1:
        d *= sign
    d = np.dot(u,d)


    rad = np.arccos(d)

    eigenvalues,eigenvectors = np.linalg.eig(T)
    
    
    
    #Find the area of an ellipse using the formula Area = major axis*minor axis* pi
    ellipseArea = np.pi*major*minor
    
    
    #Calculate the area of the contour using Shapely
    xpixels, ypixels = WCS(hdux[1].header).all_world2pix(ra,dec, 1)
    
    
    verticesPrime = np.stack((xpixels, ypixels),axis = 1)
    polygon = Polygon(verticesPrime)
    contourArea = polygon.area

    #Find the ratio between the ellipse's area and the contour's area
    areaRatio = ellipseArea/contourArea
    

    #Set the vertices around the ellipse
    shape = cutout.shape
    xnew,ynew = shape[1],shape[0]
    x,y = np.meshgrid(np.arange(xnew),np.arange(ynew))
    x, y = x.flatten(), y.flatten()
    
    xx1 = xx-xmin2
    yy1 = yy-ymin2
    

    
    ellipseVertices = np.vstack((ellipsis[0,:]+x0, ellipsis[1,:]+y0)).T
    
    
    
    

    #Grid for the ellipse, true indicates they are inside ellipse
    points = np.vstack((x,y)).T
    path = Path(ellipseVertices)
    gridEllipse = path.contains_points(points)

    #Grid for the contour, true indicates they are inside contour
   
    poly_verts = np.vstack((xx1,yy1)).T
    path = Path(poly_verts)
    gridContour = path.contains_points(points)

    
    #Ratio of ellipse's area outside of the contour to the total area of the ellipse
    #Create a blank cutout for plotting
    cutoutPrime = cutout.ravel()
    blankCutout = cutoutPrime
    blankCutout.fill(-1000)
    
    #Find the area ratio and fill out the blank cutout
    blankCutout1 = copy.deepcopy(blankCutout)
    total=0
    for index in range(len(gridEllipse)):

        if (gridEllipse[index]==True) & (gridContour[index]==False):
            total = total + 1
            blankCutout1[index] = 0
        if (gridEllipse[index] == True) & (gridContour[index] == True):
            blankCutout1[index] = 1000
        if (gridEllipse[index] == False) & (gridContour[index] == True):
            blankCutout1[index] = 1000
            
    ncols = cutout.shape[1]    

    for index in range(len(blankCutout1)):
        int(blankCutout1[index])

    blankCutout1 = np.reshape(blankCutout1, (-1, ncols))

    #Compute the area ratio

    ratio1 = total/ellipseArea
    print('The first ratio is ' + str(ratio1))

    #Find the second area ratio; this ratio is the area within the contour, but not the ellipse to the area of the contour
    #Create another blank cutout for plotting
    blankCutout2 = copy.deepcopy(blankCutout)

    #Fill out the blank cutout
    total=0
    for index in range(len(gridEllipse)):

        if (gridEllipse[index]== False) & (gridContour[index]== True):
            total = total + 1
            blankCutout2[index] = 0
        if (gridEllipse[index] == True) & (gridContour[index] == False):
            blankCutout2[index] = 1000
        if(gridEllipse[index] == True) & (gridContour[index] == True):
            blankCutout2[index] = 1000
        

            
    ncols = cutout.shape[1]    
    
    for index in range(len(blankCutout2)):
        int(blankCutout2[index])
    blankCutout2 = np.reshape(blankCutout2, (-1, ncols))
    
    #Compute the area ratio
    ratio2 = total/contourArea
    print('The second ratio is ' + str(ratio2))

   
    

    #Find the third area ratio; this ratio is the area within both the ellipse and the contour to the total area of the contour    
    #Create a blank cutout for plotting and fill it out    
    blankCutout3 = copy.deepcopy(blankCutout)
    total=0
    for index in range(len(gridEllipse)):

        if (gridEllipse[index]== True) & (gridContour[index]== True):
            total = total + 1
            blankCutout3[index] = 0
        if (gridEllipse[index] == True) & (gridContour[index] == False):
            blankCutout3[index] = 1000
        if(gridEllipse[index] == False) & (gridContour[index] == True):
            blankCutout3[index] = 1000
        

            
    ncols = cutout.shape[1]    
    for index in range(len(blankCutout3)):
        int(blankCutout3[index])
    blankCutout3 = np.reshape(blankCutout3, (-1, ncols))

    #Compute the third area ratio
    ratio3 = total/contourArea
    print('The third ratio is ' + str(ratio3))
        
      

    return ratio1, ratio2, ratio3
 

    #Additional variable definitions for clarity
    #x0,y0: light-weighted center
    #r_bar: light-weighted radius
    #major, minor: light-weighted major and minor axes
    #Use the ellipse for our calculation purposes
    
    #ellipis: light-weighted ellipse vertices
    #xx-xmin2, yy-ymin2: vertices of contour in cutout
    #zeros: cutout of contour, with pixels outside contour boundaries set to zero
    #cutout: regular cutout of contour
    #rad: position angle of light-weighted ellipse
    

In [31]:
#Define the function to compute the area ratios using a scaled light ellipse
def ScaledRatios(vertices, image ,extend):

    #Set booleans to assist with debugging
    debug = 0
    debug2 = 0


    #Calculates the isophote's light ellipse-related area ratios using a scaled light ellipse such that the ellipse has the same area as the isophote

    #Parameters
    #----------

    #vertices: list
        # the vertices of the isophote of interest

    #image: numpy.ndarray
        # the image data that computations are performed on

    #extend: int
        #value used to extend the cutout around an isophote's boundaries
        #5 is a suitable value for these calculations


    #Returns
    #---------

    #ratio1: int
        #the ratio between the ellipse's area outside of the contour to the total area of the ellipse

    #ratio2: int
        #the ratio between the contour's area not within the ellipse to the total area of the contour


    #ratio3: int
        #the ratio between the contour's area included within the ellipse to the total area of the contour

    #First, make zeros array, where all pixels outside isophote boundaries are set to zero


    #Find the maximum and minimum pixels of the image
    shape = image.shape
    #print(shape)


    xmax,xmin,ymax,ymin = (shape[1]-1),0,(shape[0]-1),0
    
    
    #Modify how the vertices are read in depending on if they are in the higher catalogs
    if verticesType == 0:

        ra = vertices[0]
        dec = vertices[1]

        xx, yy = WCS(hdux[1].header).all_world2pix(ra,dec, 1)
    
    elif verticesType == 1:
        pixx1 = vertices[:,0]
        pixy1 = vertices[:,1]

    
        ra, dec = WCS(hdux[1].header).all_pix2world(pixx1,pixy1, 1)
        xx, yy = WCS(hdux[1].header).all_world2pix(ra,dec, 1)
    else:
        print('This is not a valid verticesType value.')
        sys.exit()

    ymax1=max(yy)
    ymin1=min(yy)
    xmax1=max(xx)
    xmin1=min(xx)

    if ymax1 < 0:
        ymax1 = 0
    if ymin1 < 0:
        ymin1 = 0
    if xmax1 < 0:
        xmax1 = 0
    if xmin1 < 0:
        xmin1 =0

    if ymax1 > ymax:
        ymax1 = ymax
    if xmax1 > xmax:
        xmax1 = xmax
    
   #Find the maximum and minimum boundaries of the cutout
    ymax2 = int(np.round(ymax1)) + extend
    ymin2 = int(np.round(ymin1)) - extend
    xmax2 = int(np.round(xmax1)) + extend
    xmin2 = int(np.round(xmin1)) - extend

    cutout = image[ymin2:ymax2,xmin2:xmax2]


    poly_verts = np.vstack((xx-xmin2,yy-ymin2)).T

    shape = cutout.shape
    xnew,ynew = shape[1],shape[0]
    x,y = np.meshgrid(np.arange(xnew),np.arange(ynew))
    x, y = x.flatten(), y.flatten()

    points = np.vstack((x,y)).T
    path = Path(poly_verts)
    grid = path.contains_points(points)

    zeros = np.zeros(cutout.shape)

    grid = grid.reshape((ynew,xnew))

    zeros[grid]=cutout[grid]

    flat_array = cutout[grid]

    array = zeros
    
     #Second, find light-weighted radius

    #note: the array will be handed in as a function of index, with each corresponding to one pixel. The origin of the indexing will always be at a corner. Which corner depends on how you ask the plot to display.
    #Step 1: compute normalization, N.
    N = np.sum(zeros)
    #Step 2: compute light-weighted center.
    ny, nx = zeros.shape
    #Now create an array of x, y that is the pixel centers in the "absolute" coordinates in which it was handed in.
    x = np.arange(0.5, nx, 1)
    y = np.arange(0.5, ny, 1)
    #Now compute light-weighted centers.
    x0, y0 = np.sum(np.tensordot(x, zeros, axes = (0,1)))/N, np.sum(np.tensordot(y, zeros, axes = (0,0)))/N
    

    #Now compute Delta x and Delta y from these centers.
    Delta_x = x - x0
    Delta_y = y - y0
    #Now compute <Delta x>, <Delta y>, other guys from these centers.
    Delta_x_bar = np.sum(np.tensordot(Delta_x, array, axes = (0,1)))/N
    Delta_y_bar = np.sum(np.tensordot(Delta_y, array, axes = (0,0)))/N
    Delta_x_abs_bar = np.sum(np.tensordot(abs(Delta_x), array, axes = (0,1)))/N
    Delta_y_abs_bar = np.sum(np.tensordot(abs(Delta_y), array, axes = (0,0)))/N
    if debug:
        print('Delta_x_abs_bar, Delta_y_abs_bar = ', Delta_x_abs_bar, Delta_y_abs_bar)

    Delta_x_sq_bar = np.sum(np.tensordot(Delta_x*Delta_x, array, axes = (0,1)))/N
    Delta_y_sq_bar = np.sum(np.tensordot(Delta_y*Delta_y, array, axes = (0,0)))/N
    if debug:
        print('Delta_x_sq_bar, Delta_y_sq_bar = ', Delta_x_sq_bar, Delta_y_sq_bar)

    eps = 1e-13

    r_bar = 0.
    r_sq_bar = 0.
    cos_theta_bar = 0.
    sin_theta_bar = 0.
    tan_theta_bar = 0.
    theta_bar = 0.
    dxdy_bar = 0.

    i = 0
    for dyv in Delta_y:
        j = 0
        for dxv in Delta_x:
            if debug2:
                print('i, j =', i, j)
            if debug:
                print('i=, j=, dxv, dyv', i, j, dxv, dyv)
            r_sq = dxv*dxv + dyv*dyv
            r = np.sqrt(r_sq)
            dxdy = dxv*dyv

            #For cosine theta and sine theta.
            if r > eps:
                cos_theta = dxv/r
                #print('i=, j=, cos theta =', i, j, cos_theta)
                sin_theta = dyv/r
            else:
                cos_theta = dxv/eps
                #print('in eps area, i=, j=, cos theta =', i, j, cos_theta)
                sin_theta = dyv/eps
            #For tan theta.
            if dxv > eps:
                tan_theta = dyv/dxv
                theta = math.atan2(dyv, dxv)
                if debug2:
                    print('i=, j=,  theta =', i, j, theta)
                    print('tan theta =', tan_theta)
            else:
                tan_theta = dyv/eps
                theta = math.atan2(dyv, eps)
                if debug2:
                    print('in eps area, i=, j=, theta =', i, j,theta)
                    print('in eps area, tan_theta =', tan_theta)

            #Now apply weighting.
            r_sq *= array[i, j]
            r *= array[i, j]
            cos_theta *= array[i, j]
            #print('weighted cos theta =', cos_theta)
            sin_theta *= array[i, j]
            tan_theta *= array[i, j]
            if debug2:
                print('weighted tan theta =', theta)
            theta *= array[i, j]
            #print('weighted theta =', theta)
            #Now take averages of these obervables.
            r_sq_bar = r_sq_bar + r_sq
            r_bar = r_bar + r
            cos_theta_bar = cos_theta_bar + cos_theta
            sin_theta_bar = sin_theta_bar + sin_theta
            tan_theta_bar = tan_theta_bar + tan_theta
            theta_bar = theta_bar + theta
            dxdy_bar = dxdy_bar + dxdy

            j += 1
        i += 1

    #Now normalize all the above by N.
    r_sq_bar /= N
    r_bar /= N
    cos_theta_bar /= N
    sin_theta_bar /= N
    tan_theta_bar /= N
    theta_bar /= N
    dxdy_bar /= N

    if debug2:
        print('<r^2> =', r_sq_bar, '<r> =', r_bar,'<cos> = ',cos_theta_bar,'<sin> = ',sin_theta_bar, '<tan> = ',tan_theta_bar, '<theta> = ', theta_bar)
    #return [N, x0, y0, Delta_x_bar, Delta_y_bar, Delta_x_abs_bar, Delta_y_abs_bar, Delta_x_sq_bar, Delta_y_sq_bar, r_sq_bar, r_bar, cos_theta_bar, sin_theta_bar, tan_theta_bar, theta_bar ]
    #return r_sq_bar, r_bar #, cos_theta_bar, sin_theta_bar, tan_theta_bar, theta_bar

    xgrid,ygrid = np.meshgrid(x,y)

    #put xgrid and ygrid to have center at x0,y0
    xs = xgrid-x0
    ys = ygrid-y0

    zerosx = np.zeros(cutout.shape)
    zerosx[grid]=xs[grid]

    zerosy = np.zeros(cutout.shape)
    zerosy[grid]=ys[grid]

    # calculate inertia tensor
    T = np.zeros((2,2))
    T[0,0]=np.sum(zerosx**2*zeros)/np.sum(zeros)
    T[0,0]
    T[0,1]=np.sum(zerosx*zerosy*zeros)/np.sum(zeros)
    T[1,0]=T[0,1]
    T[1,1]=np.sum(zerosy*zerosy*zeros)/np.sum(zeros)

    eigenvalues,eigenvectors = np.linalg.eig(T)

    theta = np.linspace(0, 2*np.pi, 1000)
    ellipsis = (np.sqrt(eigenvalues) * eigenvectors) @ [np.sin(theta), np.cos(theta)]

    #print('eigenvectors ',eigenvectors)

    #first, find larger eigenvalue
    loc = np.argsort(eigenvalues)[::-1]

    major,minor = np.sqrt(eigenvalues[loc])

    # u represents axis of major axis
    u = np.array([1,0])
    d = eigenvectors[loc[0]]
    sign = np.sign(eigenvectors[1])

    if sign[0] == -1 and sign[1] == 1:
        d *= sign
    d = np.dot(u,d)


    rad = np.arccos(d)

    eigenvalues,eigenvectors = np.linalg.eig(T)
    
    #rint(ellipsis)
    
    
    #Find the area of an ellipse using the formula Area = major axis*minor axis* pi
    ellipseArea = np.pi*major*minor
    
    
    #Calculate the area of the contour using Shapely
    xpixels, ypixels = WCS(hdux[1].header).all_world2pix(ra,dec, 1)
    
    
    verticesPrime = np.stack((xpixels, ypixels),axis = 1)
    polygon = Polygon(verticesPrime)
    contourArea = polygon.area
    
    areaRatio = ellipseArea/contourArea
    
    
    shape = cutout.shape
    xnew,ynew = shape[1],shape[0]
    x,y = np.meshgrid(np.arange(xnew),np.arange(ynew))
    x, y = x.flatten(), y.flatten()
    
    xx1 = xx-xmin2
    yy1 = yy-ymin2
    
    
    #Solve for the scaling factor (R), scale the ellipse, and find relevant parameters
    
    R = math.sqrt(contourArea/ellipseArea)
    scaledEllipse = ellipsis*R
    
    verticesPrimePrime = np.stack((scaledEllipse[0,:], scaledEllipse[1,:]),axis = 1)
    polygon = Polygon(verticesPrimePrime)
    scaledEllipseArea = polygon.area
    
    ellipseScaledVertices = np.vstack((scaledEllipse[0,:]+x0, scaledEllipse[1,:]+y0)).T

    #Grid for the ellipse, true indicates they are inside ellipse
    points = np.vstack((x,y)).T
    path = Path(ellipseScaledVertices)
    gridEllipse = path.contains_points(points)

    #Grid for the contour, true indicates they are inside contour
   
    poly_verts = np.vstack((xx1,yy1)).T
    path = Path(poly_verts)
    gridContour = path.contains_points(points)

    #Find the first ratio; this ratio is the ratio of the ellipse's area outside of the contour to the total area of the ellipse
    #Define a blank cutout to be used for plotting
    cutoutPrime = cutout.ravel()
    blankCutout = cutoutPrime
    blankCutout.fill(-1000)
    
    #Solve for the area ratio and fill out the blank cutout
    blankCutout1 = copy.deepcopy(blankCutout)
    total=0
    for index in range(len(gridEllipse)):

        if (gridEllipse[index]==True) & (gridContour[index]==False):
            total = total + 1
            blankCutout1[index] = 0
        if (gridEllipse[index] == True) & (gridContour[index] == True):
            blankCutout1[index] = 1000
        if (gridEllipse[index] == False) & (gridContour[index] == True):
            blankCutout1[index] = 1000
            
    ncols = cutout.shape[1]    

    for index in range(len(blankCutout1)):
        int(blankCutout1[index])
    blankCutout1 = np.reshape(blankCutout1, (-1, ncols))
    
    #Compute the first area ratio
    ratio1 = total/scaledEllipseArea
    
    print('The first ratio is ' + str(ratio1))

    #Find the second area ratio; this is the ratio of the contour's area not included within the ellipse to the total area of the contour
    #Create a blank cutout to use for plotting
    blankCutout2 = copy.deepcopy(blankCutout)
    total=0
    for index in range(len(gridEllipse)):

        if (gridEllipse[index]== False) & (gridContour[index]== True):
            total = total + 1
            blankCutout2[index] = 0
        if (gridEllipse[index] == True) & (gridContour[index] == False):
            blankCutout2[index] = 1000
        if(gridEllipse[index] == True) & (gridContour[index] == True):
            blankCutout2[index] = 1000
        

            
    ncols = cutout.shape[1]    
    for index in range(len(blankCutout2)):
        int(blankCutout2[index])
    blankCutout2 = np.reshape(blankCutout2, (-1, ncols))
    
    #Compute the second area ratio
    ratio2 = total/contourArea

    print('The second ratio is ' + str(ratio2))

   
    

    #Find the third area ratio; this ratio is between the area within both the contour and the ellipse to the total area of the contour
    #Create a blank cutout for plotting and fill it in
        
    blankCutout3 = copy.deepcopy(blankCutout)
    total=0
    for index in range(len(gridEllipse)):

        if (gridEllipse[index]== True) & (gridContour[index]== True):
            total = total + 1
            blankCutout3[index] = 0
        if (gridEllipse[index] == True) & (gridContour[index] == False):
            blankCutout3[index] = 1000
        if(gridEllipse[index] == False) & (gridContour[index] == True):
            blankCutout3[index] = 1000
        

            
    ncols = cutout.shape[1]    
    for index in range(len(blankCutout3)):
        int(blankCutout3[index])
    blankCutout3 = np.reshape(blankCutout3, (-1, ncols))
    
    #Compute the third area ratio
    ratio3 = total/contourArea
    print('The third ratio is ' + str(ratio3))
        
    return ratio1, ratio2, ratio3
        
      


 

    #Additional variable definitions for clarity
    #x0,y0: light-weighted center
    #r_bar: light-weighted radius
    #major, minor: light-weighted major and minor axes
    #Use the ellipse for our calculation purposes
    
    #ellipis: light-weighted ellipse vertices
    #xx-xmin2, yy-ymin2: vertices of contour in cutout
    #zeros: cutout of contour, with pixels outside contour boundaries set to zero
    #cutout: regular cutout of contour
    #rad: position angle of light-weighted ellipse
    


In [33]:
#Define the function to compute the ellipticity of the contours
def EllipticityCalculator(vertices, image ,extend):

    #Set booleans to assist with debugging
    debug = 0
    debug2 = 0


    #Calculates the isophote's ellipticity using select areas

    #Parameters
    #----------

    #vertices: list
        # the vertices of the isophote of interest

    #image: numpy.ndarray
        # the image data that computations are performed on

    #extend: int
        #value used to extend the cutout around an isophote's boundaries
        #5 is a suitable value for these calculations

    #Returns
    #-----------

    #ellipticity: int
        #the ellipticity of the isophote of interest
    

    #First, make zeros array, where all pixels outside isophote boundaries are set to zero


    #Set the boundaries of the image
    shape = image.shape

    xmax,xmin,ymax,ymin = (shape[1]-1),0,(shape[0]-1),0

    #Modify how the contour vertices are read-in depending on if they are part of the higher catalogs
    if verticesType == 0:

        ra = vertices[0]
        dec = vertices[1]

        xx, yy = WCS(hdux[1].header).all_world2pix(ra,dec, 1)
    
    elif verticesType == 1:
        pixx1 = vertices[:,0]
        pixy1 = vertices[:,1]

    
        ra, dec = WCS(hdux[1].header).all_pix2world(pixx1,pixy1, 1)
        xx, yy = WCS(hdux[1].header).all_world2pix(ra,dec, 1)
    else:
        print('This is not a valid verticesType value.')
        sys.exit()

    
    ymax1=max(yy)
    ymin1=min(yy)
    xmax1=max(xx)
    xmin1=min(xx)

    if ymax1 < 0:
        ymax1 = 0
    if ymin1 < 0:
        ymin1 = 0
    if xmax1 < 0:
        xmax1 = 0
    if xmin1 < 0:
        xmin1 =0

    if ymax1 > ymax:
        ymax1 = ymax
    if xmax1 > xmax:
        xmax1 = xmax

   #Set the boundaries of the cutout around the isophote
    ymax2 = int(np.round(ymax1)) + extend
    ymin2 = int(np.round(ymin1)) - extend
    xmax2 = int(np.round(xmax1)) + extend
    xmin2 = int(np.round(xmin1)) - extend
   
    
    
    #Find the cutout around the isophote
    cutout = image[ymin2:ymax2,xmin2:xmax2]

    poly_verts = np.vstack((xx-xmin2,yy-ymin2)).T

    shape = cutout.shape
    xnew,ynew = shape[1],shape[0]
    x,y = np.meshgrid(np.arange(xnew),np.arange(ynew))
    x, y = x.flatten(), y.flatten()

    points = np.vstack((x,y)).T
    path = Path(poly_verts)
    grid = path.contains_points(points)

    zeros = np.zeros(cutout.shape)

    grid = grid.reshape((ynew,xnew))

    zeros[grid]=cutout[grid]

    flat_array = cutout[grid]

    array = zeros
    
     #Second, find light-weighted radius

    #note: the array will be handed in as a function of index, with each corresponding to one pixel. The origin of the indexing will always be at a corner. Which corner depends on how you ask the plot to display.
    #Step 1: compute normalization, N.
    N = np.sum(zeros)
    #Step 2: compute light-weighted center.
    ny, nx = zeros.shape
    #Now create an array of x, y that is the pixel centers in the "absolute" coordinates in which it was handed in.
    x = np.arange(0.5, nx, 1)
    y = np.arange(0.5, ny, 1)
    #Now compute light-weighted centers.
    x0, y0 = np.sum(np.tensordot(x, zeros, axes = (0,1)))/N, np.sum(np.tensordot(y, zeros, axes = (0,0)))/N
    

    #Now compute Delta x and Delta y from these centers.
    Delta_x = x - x0
    Delta_y = y - y0
    #Now compute <Delta x>, <Delta y>, other guys from these centers.
    Delta_x_bar = np.sum(np.tensordot(Delta_x, array, axes = (0,1)))/N
    Delta_y_bar = np.sum(np.tensordot(Delta_y, array, axes = (0,0)))/N
    Delta_x_abs_bar = np.sum(np.tensordot(abs(Delta_x), array, axes = (0,1)))/N
    Delta_y_abs_bar = np.sum(np.tensordot(abs(Delta_y), array, axes = (0,0)))/N
    if debug:
        print('Delta_x_abs_bar, Delta_y_abs_bar = ', Delta_x_abs_bar, Delta_y_abs_bar)

    Delta_x_sq_bar = np.sum(np.tensordot(Delta_x*Delta_x, array, axes = (0,1)))/N
    Delta_y_sq_bar = np.sum(np.tensordot(Delta_y*Delta_y, array, axes = (0,0)))/N
    if debug:
        print('Delta_x_sq_bar, Delta_y_sq_bar = ', Delta_x_sq_bar, Delta_y_sq_bar)

    eps = 1e-13

    r_bar = 0.
    r_sq_bar = 0.
    cos_theta_bar = 0.
    sin_theta_bar = 0.
    tan_theta_bar = 0.
    theta_bar = 0.
    dxdy_bar = 0.

    i = 0
    for dyv in Delta_y:
        j = 0
        for dxv in Delta_x:
            if debug2:
                print('i, j =', i, j)
            if debug:
                print('i=, j=, dxv, dyv', i, j, dxv, dyv)
            r_sq = dxv*dxv + dyv*dyv
            r = np.sqrt(r_sq)
            dxdy = dxv*dyv

            #For cosine theta and sine theta.
            if r > eps:
                cos_theta = dxv/r
                #print('i=, j=, cos theta =', i, j, cos_theta)
                sin_theta = dyv/r
            else:
                cos_theta = dxv/eps
                #print('in eps area, i=, j=, cos theta =', i, j, cos_theta)
                sin_theta = dyv/eps
            #For tan theta.
            if dxv > eps:
                tan_theta = dyv/dxv
                theta = math.atan2(dyv, dxv)
                if debug2:
                    print('i=, j=,  theta =', i, j, theta)
                    print('tan theta =', tan_theta)
            else:
                tan_theta = dyv/eps
                theta = math.atan2(dyv, eps)
                if debug2:
                    print('in eps area, i=, j=, theta =', i, j,theta)
                    print('in eps area, tan_theta =', tan_theta)

            #Now apply weighting.
            r_sq *= array[i, j]
            r *= array[i, j]
            cos_theta *= array[i, j]
            #print('weighted cos theta =', cos_theta)
            sin_theta *= array[i, j]
            tan_theta *= array[i, j]
            if debug2:
                print('weighted tan theta =', theta)
            theta *= array[i, j]
            #print('weighted theta =', theta)
            #Now take averages of these obervables.
            r_sq_bar = r_sq_bar + r_sq
            r_bar = r_bar + r
            cos_theta_bar = cos_theta_bar + cos_theta
            sin_theta_bar = sin_theta_bar + sin_theta
            tan_theta_bar = tan_theta_bar + tan_theta
            theta_bar = theta_bar + theta
            dxdy_bar = dxdy_bar + dxdy

            j += 1
        i += 1

    #Now normalize all the above by N.
    r_sq_bar /= N
    r_bar /= N
    cos_theta_bar /= N
    sin_theta_bar /= N
    tan_theta_bar /= N
    theta_bar /= N
    dxdy_bar /= N

    if debug2:
        print('<r^2> =', r_sq_bar, '<r> =', r_bar,'<cos> = ',cos_theta_bar,'<sin> = ',sin_theta_bar, '<tan> = ',tan_theta_bar, '<theta> = ', theta_bar)
    #return [N, x0, y0, Delta_x_bar, Delta_y_bar, Delta_x_abs_bar, Delta_y_abs_bar, Delta_x_sq_bar, Delta_y_sq_bar, r_sq_bar, r_bar, cos_theta_bar, sin_theta_bar, tan_theta_bar, theta_bar ]
    #return r_sq_bar, r_bar #, cos_theta_bar, sin_theta_bar, tan_theta_bar, theta_bar

    xgrid,ygrid = np.meshgrid(x,y)

    #put xgrid and ygrid to have center at x0,y0
    xs = xgrid-x0
    ys = ygrid-y0

    zerosx = np.zeros(cutout.shape)
    zerosx[grid]=xs[grid]

    zerosy = np.zeros(cutout.shape)
    zerosy[grid]=ys[grid]

    # calculate inertia tensor
    T = np.zeros((2,2))
    T[0,0]=np.sum(zerosx**2*zeros)/np.sum(zeros)
    T[0,0]
    T[0,1]=np.sum(zerosx*zerosy*zeros)/np.sum(zeros)
    T[1,0]=T[0,1]
    T[1,1]=np.sum(zerosy*zerosy*zeros)/np.sum(zeros)

    eigenvalues,eigenvectors = np.linalg.eig(T)

    theta = np.linspace(0, 2*np.pi, 1000)
    ellipsis = (np.sqrt(eigenvalues) * eigenvectors) @ [np.sin(theta), np.cos(theta)]

    #print('eigenvectors ',eigenvectors)

    #first, find larger eigenvalue
    loc = np.argsort(eigenvalues)[::-1]

    major,minor = np.sqrt(eigenvalues[loc])

    # u represents axis of major axis
    u = np.array([1,0])
    d = eigenvectors[loc[0]]
    sign = np.sign(eigenvectors[1])

    if sign[0] == -1 and sign[1] == 1:
        d *= sign
    d = np.dot(u,d)


    rad = np.arccos(d)

    eigenvalues,eigenvectors = np.linalg.eig(T)
    
    #rint(ellipsis)
    
    
    #Find the area of an ellipse using the formula Area = major axis*minor axis* pi
    ellipseArea = np.pi*major*minor
    
    
    #Calculate the area of the contour using Shapely
    xpixels, ypixels = WCS(hdux[1].header).all_world2pix(ra,dec, 1)
    
    
    verticesPrime = np.stack((xpixels, ypixels),axis = 1)
    polygon = Polygon(verticesPrime)
    contourArea = polygon.area
    
    areaRatio = ellipseArea/contourArea
    
    
    shape = cutout.shape
    xnew,ynew = shape[1],shape[0]
    x,y = np.meshgrid(np.arange(xnew),np.arange(ynew))
    x, y = x.flatten(), y.flatten()
    
    xx1 = xx-xmin2
    yy1 = yy-ymin2
    
    
    #Solve for the scaling factor (R), scale the ellipse, and solve for relevant parameters
    
    R = math.sqrt(contourArea/ellipseArea)
    scaledEllipse = ellipsis*R
    
    verticesPrimePrime = np.stack((scaledEllipse[0,:], scaledEllipse[1,:]),axis = 1)
    polygon = Polygon(verticesPrimePrime)
    scaledEllipseArea = polygon.area
    
    ellipseScaledVertices = np.vstack((scaledEllipse[0,:]+x0, scaledEllipse[1,:]+y0)).T

    #Grid for the ellipse, true indicates they are inside ellipse
    points = np.vstack((x,y)).T
    path = Path(ellipseScaledVertices)
    gridEllipse = path.contains_points(points)

    #Grid for the contour, true indicates they are inside contour
   
    poly_verts = np.vstack((xx1,yy1)).T
    path = Path(poly_verts)
    gridContour = path.contains_points(points)
  
    #Area within the ellipse but outside of the contour
    #Create a blank cutout to help plot the area of interest

    cutoutPrime = cutout.ravel()
    blankCutout = cutoutPrime
    blankCutout.fill(-1000)
    
    #Find the area of interest and fill in the cutout
    blankCutout1 = copy.deepcopy(blankCutout)
    total=0
    for index in range(len(gridEllipse)):

        if (gridEllipse[index]==True) & (gridContour[index]==False):
            total = total + 1
            blankCutout1[index] = 0
        if (gridEllipse[index] == True) & (gridContour[index] == True):
            blankCutout1[index] = 1000
        if (gridEllipse[index] == False) & (gridContour[index] == True):
            blankCutout1[index] = 1000
            
    ncols = cutout.shape[1]    
    for index in range(len(blankCutout1)):
        int(blankCutout1[index])
        
    blankCutout1 = np.reshape(blankCutout1, (-1, ncols))

    #Set area variables to be used in the ellipticity calculation
    total1 = total
    area1 = total1
    
    
    #Find the area within the contour but not the ellipse
    #Create a blank cutout and fill it in for plotting purposes
    blankCutout2 = copy.deepcopy(blankCutout)
    total=0


    for index in range(len(gridEllipse)):

        if (gridEllipse[index]== False) & (gridContour[index]== True):
            total = total + 1
            blankCutout2[index] = 0
        if (gridEllipse[index] == True) & (gridContour[index] == False):
            blankCutout2[index] = 1000
        if(gridEllipse[index] == True) & (gridContour[index] == True):
            blankCutout2[index] = 1000

            
    ncols = cutout.shape[1]    
    for index in range(len(blankCutout2)):
        int(blankCutout2[index])

    blankCutout2 = np.reshape(blankCutout2, (-1, ncols))

    #Set variables for the area to use in the ellipticity calculation
    total2 = total
    area2 = total2

    #Compute the ellipticity
    ellipticity = 1 - ((area2 + area1)/(2*contourArea))
        
    print('The ellipticity is: ' + str(ellipticity))

        
      

    return ellipticity
 

    #Additional variable definitions for clarity
    #x0,y0: light-weighted center
    #r_bar: light-weighted radius
    #major, minor: light-weighted major and minor axes
    #Use the ellipse for our calculation purposes
    
    #ellipis: light-weighted ellipse vertices
    #xx-xmin2, yy-ymin2: vertices of contour in cutout
    #zeros: cutout of contour, with pixels outside contour boundaries set to zero
    #cutout: regular cutout of contour
    #rad: position angle of light-weighted ellipse
    


In [ ]:
#Find the desired parameters for the conservative catalog
#Create a dictionary to store values
conservativeAreaRatios = {}
conservativeAreaRatios['key'] = []

conservativeAreaRatios['Unscaled EllipseOutsideContour Over TotalEllipse'] = []
conservativeAreaRatios['Unscaled IsophoteOutsideEllipse Over TotalIsophote'] = []
conservativeAreaRatios['Unscaled InsideBoth Over TotalIsophote'] = []

conservativeAreaRatios['Scaled EllipseOutsideContour Over TotalEllipse'] = []
conservativeAreaRatios['Scaled IsophoteOutsideEllipse Over TotalIsophote'] = []
conservativeAreaRatios['Scaled InsideBoth Over TotalIsophote'] = []

conservativeAreaRatios['Ellipticity'] = []

#Loop over the vertices in the catalog, compute relevant parameters, and save them to a dictionary
for key in conservativeVertices:
    verticesType = 0
    vertices = conservativeVertices.get(key)
    
    conservativeAreaRatios['key'].append(key)
    
    unscaledValues = UnscaledRatios(vertices, usefulImagePrime, extend)
    scaledValues = ScaledRatios(vertices, usefulImagePrime, extend)
    ellipticity = EllipticityCalculator(vertices, usefulImagePrime, extend)

    unscaledArea1 = unscaledValues[0]
    unscaledArea2 = unscaledValues[1]
    unscaledArea3 = unscaledValues[2]
    
    scaledArea1 = scaledValues[0]
    scaledArea2 = scaledValues[1]
    scaledArea3 = scaledValues[2]
    
    conservativeAreaRatios['Unscaled EllipseOutsideContour Over TotalEllipse'].append(unscaledArea1)
    conservativeAreaRatios['Unscaled IsophoteOutsideEllipse Over TotalIsophote'].append(unscaledArea2)
    conservativeAreaRatios['Unscaled InsideBoth Over TotalIsophote'].append(unscaledArea3)
    
    conservativeAreaRatios['Scaled EllipseOutsideContour Over TotalEllipse'].append(scaledArea1)
    conservativeAreaRatios['Scaled IsophoteOutsideEllipse Over TotalIsophote'].append(scaledArea2)
    conservativeAreaRatios['Scaled InsideBoth Over TotalIsophote'].append(scaledArea3)
    
    conservativeAreaRatios['Ellipticity'].append(ellipticity)

#Print the relevant parameters to a table and save as a FITS file
conservativeAreaRatios_table = Table(conservativeAreaRatios)
print(conservativeAreaRatios_table)   
conservativeAreaRatios_table.write('ConservativeAreaRatios(UPDATED_3-26-25).fits')
    

In [ ]:
#Find the relevant parameters for the expansive catalog
#Create a dictionary to store values
expansiveAreaRatios = {}
expansiveAreaRatios['key'] = []

expansiveAreaRatios['Unscaled EllipseOutsideContour Over TotalEllipse'] = []
expansiveAreaRatios['Unscaled IsophoteOutsideEllipse Over TotalIsophote'] = []
expansiveAreaRatios['Unscaled InsideBoth Over TotalIsophote'] = []

expansiveAreaRatios['Scaled EllipseOutsideContour Over TotalEllipse'] = []
expansiveAreaRatios['Scaled IsophoteOutsideEllipse Over TotalIsophote'] = []
expansiveAreaRatios['Scaled InsideBoth Over TotalIsophote'] = []

expansiveAreaRatios['Ellipticity'] = []

#Loop over the vertices, compute the desired parameters, and save them to a dictionary
for key in expansiveVertices:
    vertices = expansiveVertices.get(key)
    verticesType = 0
    
    expansiveAreaRatios['key'].append(key)
    
    unscaledValues = UnscaledRatios(vertices, usefulImagePrime, extend)
    scaledValues = ScaledRatios(vertices, usefulImagePrime, extend)
    ellipticity = EllipticityCalculator(vertices, usefulImagePrime, extend)

    unscaledArea1 = unscaledValues[0]
    unscaledArea2 = unscaledValues[1]
    unscaledArea3 = unscaledValues[2]
    
    scaledArea1 = scaledValues[0]
    scaledArea2 = scaledValues[1]
    scaledArea3 = scaledValues[2]
    
    expansiveAreaRatios['Unscaled EllipseOutsideContour Over TotalEllipse'].append(unscaledArea1)
    expansiveAreaRatios['Unscaled IsophoteOutsideEllipse Over TotalIsophote'].append(unscaledArea2)
    expansiveAreaRatios['Unscaled InsideBoth Over TotalIsophote'].append(unscaledArea3)
    
    expansiveAreaRatios['Scaled EllipseOutsideContour Over TotalEllipse'].append(scaledArea1)
    expansiveAreaRatios['Scaled IsophoteOutsideEllipse Over TotalIsophote'].append(scaledArea2)
    expansiveAreaRatios['Scaled InsideBoth Over TotalIsophote'].append(scaledArea3)
    
    expansiveAreaRatios['Ellipticity'].append(ellipticity)

#Output the relevant parameters as a table and save as a FITS file
expansiveAreaRatios_table = Table(expansiveAreaRatios)
print(expansiveAreaRatios_table)   
expansiveAreaRatios_table.write('ExpansiveAreaRatios(UPDATED_2-11-25).fits')

In [9]:
#Load the faulty contours in the higher catalogs 
faultyConservativeHigher = np.load(r"C:\Users\jacob\AstronomyResearchStuff\sf1.npy")

In [ ]:
#Find the relevant values in the conservative higher catalog
#Create a dictionary to store values
conservativeHigherAreaRatios = {}
conservativeHigherAreaRatios['key'] = []

conservativeHigherAreaRatios['Unscaled EllipseOutsideContour Over TotalEllipse'] = []
conservativeHigherAreaRatios['Unscaled IsophoteOutsideEllipse Over TotalIsophote'] = []
conservativeHigherAreaRatios['Unscaled InsideBoth Over TotalIsophote'] = []

conservativeHigherAreaRatios['Scaled EllipseOutsideContour Over TotalEllipse'] = []
conservativeHigherAreaRatios['Scaled IsophoteOutsideEllipse Over TotalIsophote'] = []
conservativeHigherAreaRatios['Scaled InsideBoth Over TotalIsophote'] = []

conservativeHigherAreaRatios['Ellipticity'] = []

#Loop over the vertices, compute the relevant parameters, and save to a dictionary
for key in conservativeHigherVertices:
    vertices = conservativeHigherVertices.get(key)
    verticesType = 1
    
    print('The key is: ' + str(key))
    
    if key in faultyConservativeHigher:
        continue
    
    conservativeHigherAreaRatios['key'].append(key)
    
    unscaledValues = UnscaledRatios(vertices, usefulImagePrime, extend)
    scaledValues = ScaledRatios(vertices, usefulImagePrime, extend)
    ellipticity = EllipticityCalculator(vertices, usefulImagePrime, extend)

    unscaledArea1 = unscaledValues[0]
    unscaledArea2 = unscaledValues[1]
    unscaledArea3 = unscaledValues[2]
    
    scaledArea1 = scaledValues[0]
    scaledArea2 = scaledValues[1]
    scaledArea3 = scaledValues[2]
    
    conservativeHigherAreaRatios['Unscaled EllipseOutsideContour Over TotalEllipse'].append(unscaledArea1)
    conservativeHigherAreaRatios['Unscaled IsophoteOutsideEllipse Over TotalIsophote'].append(unscaledArea2)
    conservativeHigherAreaRatios['Unscaled InsideBoth Over TotalIsophote'].append(unscaledArea3)
    
    conservativeHigherAreaRatios['Scaled EllipseOutsideContour Over TotalEllipse'].append(scaledArea1)
    conservativeHigherAreaRatios['Scaled IsophoteOutsideEllipse Over TotalIsophote'].append(scaledArea2)
    conservativeHigherAreaRatios['Scaled InsideBoth Over TotalIsophote'].append(scaledArea3)
    
    conservativeHigherAreaRatios['Ellipticity'].append(ellipticity)

#Output the parameters as a table and save as a FITS file
conservativeHigherAreaRatios_table = Table(conservativeHigherAreaRatios)
print(conservativeHigherAreaRatios_table)   
conservativeHigherAreaRatios_table.write('ConservativeHigherAreaRatios(UPDATED_2-11-25).fits')
    

In [ ]:
#Find the parameters for the expansive higher catalog
#Create a dictionary to store values
expansiveHigherAreaRatios = {}
expansiveHigherAreaRatios['key'] = []

expansiveHigherAreaRatios['Unscaled EllipseOutsideContour Over TotalEllipse'] = []
expansiveHigherAreaRatios['Unscaled IsophoteOutsideEllipse Over TotalIsophote'] = []
expansiveHigherAreaRatios['Unscaled InsideBoth Over TotalIsophote'] = []

expansiveHigherAreaRatios['Scaled EllipseOutsideContour Over TotalEllipse'] = []
expansiveHigherAreaRatios['Scaled IsophoteOutsideEllipse Over TotalIsophote'] = []
expansiveHigherAreaRatios['Scaled InsideBoth Over TotalIsophote'] = []

expansiveHigherAreaRatios['Ellipticity'] = []

#Loop over the vertices, compute the relevant parameters, and save to a dictionary
for key in expansiveHigherVertices:
    vertices = expansiveHigherVertices.get(key)
    verticesType = 1
    
    expansiveHigherAreaRatios['key'].append(key)
    
    unscaledValues = UnscaledRatios(vertices, usefulImagePrime, extend)
    scaledValues = ScaledRatios(vertices, usefulImagePrime, extend)
    ellipticity = EllipticityCalculator(vertices, usefulImagePrime, extend)

    unscaledArea1 = unscaledValues[0]
    unscaledArea2 = unscaledValues[1]
    unscaledArea3 = unscaledValues[2]
    
    scaledArea1 = scaledValues[0]
    scaledArea2 = scaledValues[1]
    scaledArea3 = scaledValues[2]
    
    expansiveHigherAreaRatios['Unscaled EllipseOutsideContour Over TotalEllipse'].append(unscaledArea1)
    expansiveHigherAreaRatios['Unscaled IsophoteOutsideEllipse Over TotalIsophote'].append(unscaledArea2)
    expansiveHigherAreaRatios['Unscaled InsideBoth Over TotalIsophote'].append(unscaledArea3)
    
    expansiveHigherAreaRatios['Scaled EllipseOutsideContour Over TotalEllipse'].append(scaledArea1)
    expansiveHigherAreaRatios['Scaled IsophoteOutsideEllipse Over TotalIsophote'].append(scaledArea2)
    expansiveHigherAreaRatios['Scaled InsideBoth Over TotalIsophote'].append(scaledArea3)
    
    expansiveHigherAreaRatios['Ellipticity'].append(ellipticity)

#Output the parameters to a table and save as a FITS file
expansiveHigherAreaRatios_table = Table(expansiveHigherAreaRatios)
print(expansiveHigherAreaRatios_table)   
expansiveHigherAreaRatios_table.write('ExpansiveHigherAreaRatios(UPDATED_2-11-25).fits')